In [ ]:
!pip install -q scikit-learn==1.9.1

In [ ]:
from google.colab import files
uploaded = files.upload()     # click "Choose files" and select placement.csv from D:\careersync-ai
print(list(uploaded.keys()))

Saving placementdata.csv to placementdata.csv
['placementdata.csv']


In [ ]:
import sys
import joblib
import pandas as pd
import sklearn
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

print("scikit-learn version:", sklearn.__version__)   # must say 1.9.1

CSV_NAME = "placementdata.csv"
df = pd.read_csv(CSV_NAME)
print("Rows:", len(df))
print("Columns:", df.columns.tolist())

FEATURES = [
    "CGPA", "Internships", "Projects", "Workshops/Certifications",
    "SSC_Marks", "HSC_Marks", "ExtracurricularActivities", "PlacementTraining",
]
TARGET = "PlacementStatus"

missing = [c for c in FEATURES + [TARGET] if c not in df.columns]
if missing:
    raise SystemExit(f"STOP: missing columns {missing}. Send me the 'Columns:' line above.")

y = df[TARGET]
if y.dtype != object:
    y = y.map({1: "Placed", 0: "NotPlaced"})
print("Labels:", y.value_counts().to_dict())
if "Placed" not in set(y):
    raise SystemExit("STOP: no 'Placed' label found. Send me the 'Labels:' line above.")

X = df[FEATURES]
cat = ["ExtracurricularActivities", "PlacementTraining"]
num = [c for c in FEATURES if c not in cat]

pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat),
    ("num", StandardScaler(), num),
])

candidates = {
    "logistic_regression": LogisticRegression(max_iter=1000),
    "random_forest": RandomForestClassifier(n_estimators=300, random_state=42),
    "neural_network_mlp": MLPClassifier(
        hidden_layer_sizes=(32, 16), early_stopping=True,
        max_iter=500, random_state=42,
    ),
}

Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

best_name, best_pipe, best_score = None, None, -1
for name, clf in candidates.items():
    pipe = Pipeline([("pre", clone(pre)), ("clf", clf)])
    pipe.fit(Xtr, ytr)
    score = pipe.score(Xte, yte)
    print(f"{name}: accuracy = {score:.4f}")
    if score > best_score:
        best_name, best_pipe, best_score = name, pipe, score

print(f"\nBest model: {best_name} ({best_score:.4f})")
print("Classes:", list(best_pipe.classes_))

joblib.dump(best_pipe, "placement_model.joblib")
from google.colab import files
files.download("placement_model.joblib")

scikit-learn version: 1.9.1
Rows: 10000
Columns: ['StudentID', 'CGPA', 'Internships', 'Projects', 'Workshops/Certifications', 'AptitudeTestScore', 'SoftSkillsRating', 'ExtracurricularActivities', 'PlacementTraining', 'SSC_Marks', 'HSC_Marks', 'PlacementStatus']
Labels: {'NotPlaced': 5803, 'Placed': 4197}
logistic_regression: accuracy = 0.8015
random_forest: accuracy = 0.7780
neural_network_mlp: accuracy = 0.8005

Best model: logistic_regression (0.8015)
Classes: ['NotPlaced', 'Placed']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>